In [4]:
#version de python
import sys
sys.version

'3.11.11 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:34:19) [MSC v.1929 64 bit (AMD64)]'

In [5]:
#version de pycaret
import pycaret
print(pycaret.__version__)

3.2.0


In [6]:
#version de mlflow
import mlflow
print(mlflow.__version__)

1.30.1


Importation des données

In [7]:
#changer de dossier
import os
os.chdir("C:/Users/lucie/test-venv")

In [8]:
import mlflow
mlflow.set_tracking_uri("file:///C:/Users/lucie/test-venv/mlruns")

In [9]:
#chargement des données 
import pandas as pd
data = pd.read_csv('Loan_Data.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               10000 non-null  int64  
 1   credit_lines_outstanding  10000 non-null  int64  
 2   loan_amt_outstanding      10000 non-null  float64
 3   total_debt_outstanding    10000 non-null  float64
 4   income                    10000 non-null  float64
 5   years_employed            10000 non-null  int64  
 6   fico_score                10000 non-null  int64  
 7   default                   10000 non-null  int64  
dtypes: float64(3), int64(5)
memory usage: 625.1 KB


Comparaison des algorithmes

In [10]:
#importation de l'outil d'expérimentation de pycaret
from pycaret.classification import ClassificationExperiment

In [11]:
#création d'une session de travail
session_prim = ClassificationExperiment()
session_prim.setup(data,normalize=True,target='default',train_size=0.7,
                   data_split_stratify=True,fold=5,session_id=0,
                   log_experiment=True,experiment_name="recherche_modele")

,Description,Value
0,Session id,0
1,Target,default
2,Target type,Binary
3,Original data shape,"(10000, 8)"
4,Transformed data shape,"(10000, 8)"
5,Transformed train set shape,"(7000, 8)"
6,Transformed test set shape,"(3000, 8)"
7,Numeric features,7
8,Preprocess,True
9,Imputation type,simple


2025/02/13 15:32:31 INFO mlflow.tracking.fluent: Experiment with name 'recherche_modele' does not exist. Creating a new experiment.


In [12]:
#algorithmes disponibles pour l'environnment crée
algos=session_prim.models()
print(algos)

                                     Name  \
ID                                          
lr                    Logistic Regression   
knn                K Neighbors Classifier   
nb                            Naive Bayes   
dt               Decision Tree Classifier   
svm                   SVM - Linear Kernel   
rbfsvm                SVM - Radial Kernel   
gpc           Gaussian Process Classifier   
mlp                        MLP Classifier   
ridge                    Ridge Classifier   
rf               Random Forest Classifier   
qda       Quadratic Discriminant Analysis   
ada                  Ada Boost Classifier   
gbc          Gradient Boosting Classifier   
lda          Linear Discriminant Analysis   
et                 Extra Trees Classifier   
lightgbm  Light Gradient Boosting Machine   
dummy                    Dummy Classifier   

                                                  Reference  Turbo  
ID                                                                  
lr    

In [13]:
#comparer et sélectionner les modèles
top_models = session_prim.compare_models(sort='Accuracy', include=['lr','nb','dt','rf','svm','lda'], verbose=True)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.9976,1.0000,0.9884,0.9984,0.9934,0.9919,0.9919,1.8960
svm,SVM - Linear Kernel,0.9969,0.0000,0.9892,0.9939,0.9914,0.9895,0.9896,0.0280
dt,Decision Tree Classifier,0.9943,0.9914,0.9869,0.9824,0.9846,0.9811,0.9811,0.0480
rf,Random Forest Classifier,0.9939,0.9998,0.9807,0.9861,0.9834,0.9796,0.9796,0.2780
lda,Linear Discriminant Analysis,0.9787,0.9997,0.9992,0.8976,0.9457,0.9325,0.9346,0.0280
nb,Naive Bayes,0.9721,0.9973,0.9722,0.8880,0.9282,0.9109,0.9124,1.8560


In [14]:
#obtenir le détail des résultats
session_prim.pull()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.9976,1.0000,0.9884,0.9984,0.9934,0.9919,0.9919,1.896
svm,SVM - Linear Kernel,0.9969,0.0000,0.9892,0.9939,0.9914,0.9895,0.9896,0.028
dt,Decision Tree Classifier,0.9943,0.9914,0.9869,0.9824,0.9846,0.9811,0.9811,0.048
rf,Random Forest Classifier,0.9939,0.9998,0.9807,0.9861,0.9834,0.9796,0.9796,0.278
lda,Linear Discriminant Analysis,0.9787,0.9997,0.9992,0.8976,0.9457,0.9325,0.9346,0.028
nb,Naive Bayes,0.9721,0.9973,0.9722,0.8880,0.9282,0.9109,0.9124,1.856


Optimisation des paramètres - Arbre de décision

In [15]:
session_bis = ClassificationExperiment()
session_bis.setup(data,normalize=True,target='default',train_size=0.7,
                 data_split_stratify=True,fold=5,session_id=0,
                 log_experiment=True,experiment_name="optimisation_modele")

,Description,Value
0,Session id,0
1,Target,default
2,Target type,Binary
3,Original data shape,"(10000, 8)"
4,Transformed data shape,"(10000, 8)"
5,Transformed train set shape,"(7000, 8)"
6,Transformed test set shape,"(3000, 8)"
7,Numeric features,7
8,Preprocess,True
9,Imputation type,simple


2025/02/13 15:32:59 INFO mlflow.tracking.fluent: Experiment with name 'optimisation_modele' does not exist. Creating a new experiment.


In [16]:
#instantiation de l'arbre + premier paramétrage
mybest=session_bis.create_model("dt", min_samples_split=50, max_depth=5)
print(mybest)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9871,0.9983,0.9614,0.9689,0.9651,0.9572,0.9572
1,0.9879,0.9975,0.9575,0.9764,0.9669,0.9594,0.9595
2,0.9900,0.9990,0.9846,0.9623,0.9733,0.9671,0.9672
3,0.9893,0.9995,0.9421,1.0000,0.9702,0.9637,0.9643
4,0.9921,0.9968,0.9692,0.9882,0.9786,0.9738,0.9739
Mean,0.9893,0.9982,0.9630,0.9791,0.9708,0.9643,0.9644
Std,0.0017,0.0010,0.0140,0.0135,0.0048,0.0059,0.0059


DecisionTreeClassifier(ccp_alpha=0.0, class_weight=None, criterion='gini',
                       max_depth=5, max_features=None, max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_samples_leaf=1,
                       min_samples_split=50, min_weight_fraction_leaf=0.0,
                       random_state=0, splitter='best')


In [17]:
#optimisation des paramètres de l'arbre
tuned_mybest, essais = session_bis.tune_model(mybest, optimize="Accuracy",
                       choose_better=True, custom_grid={'min_samples_split':
                       [2,10,20], 'max_depth':[5,10,None]}, search_algorithm='grid', return_tuner=True)  

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9943,0.9924,0.9846,0.9846,0.9846,0.9811,0.9811
1,0.9943,0.9966,0.9884,0.9808,0.9846,0.9811,0.9811
2,0.9929,0.9927,0.9807,0.9807,0.9807,0.9763,0.9763
3,0.9964,0.9952,0.9884,0.9922,0.9903,0.9881,0.9881
4,0.9957,0.9952,0.9885,0.9885,0.9885,0.9858,0.9858
Mean,0.9947,0.9944,0.9861,0.9854,0.9857,0.9825,0.9825
Std,0.0012,0.0016,0.0031,0.0045,0.0034,0.0041,0.0041


Fitting 5 folds for each of 9 candidates, totalling 45 fits


In [18]:
#affichage des essais
import pandas as pd
pd.DataFrame.from_dict(essais.cv_results_)[
  ['param_actual_estimator__max_depth', 'param_actual_estimator__min_samples_split', 'mean_test_score']
].sort_values(by='mean_test_score', ascending=False)

,param_actual_estimator__max_depth,param_actual_estimator__min_samples_split,mean_test_score
4,10,10,0.994714
7,None,10,0.994571
6,None,2,0.994286
1,5,10,0.994000
0,5,2,0.994000
3,10,2,0.994000
2,5,20,0.993143
5,10,20,0.993143
8,None,20,0.993000


In [19]:
#caractéristiques du modèle "optimisé"
print(tuned_mybest)

DecisionTreeClassifier(ccp_alpha=0.0, class_weight=None, criterion='gini',
                       max_depth=10, max_features=None, max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_samples_leaf=1,
                       min_samples_split=10, min_weight_fraction_leaf=0.0,
                       random_state=0, splitter='best')


In [20]:
#évaluation de l'échantillon test
#les 30% (1.0 - 0.7) mis de coté au préalable
#lors de l'initialisation de la session
session_bis.predict_model(tuned_mybest)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Decision Tree Classifier,0.9933,0.9888,0.9748,0.9890,0.9819,0.9778,0.9778


,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default,prediction_label,prediction_score
8939,6003318,1,4919.171387,8299.041992,75203.265625,5,644,0,0,1.0
1088,4262157,1,4656.162109,7578.003418,84063.617188,5,653,0,0,1.0
8346,5315847,0,4597.693359,7551.521973,80341.835938,4,610,0,0,1.0
26,7269476,0,3040.960938,4404.918457,47012.429688,2,636,0,0,1.0
5775,5166893,1,3795.419922,7627.235352,55330.714844,5,622,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...
9122,5510443,1,3550.871094,7398.292969,60806.882812,5,673,0,0,1.0
9619,1711980,0,5771.317871,4362.190430,105356.281250,6,693,0,0,1.0
8520,8290510,0,5089.931641,789.115356,78911.539062,5,648,0,0,1.0
4529,7790604,1,3598.088623,5802.745117,56944.613281,5,669,0,0,1.0


In [21]:
#modèle définitif pour le déploiement
#le modèle est ré-entrainésur la totalité
#des données disponibles
modele_definitif=session_bis.finalize_model(tuned_mybest)
print(modele_definitif)

Pipeline(memory=Memory(location=None),
         steps=[('numerical_imputer',
                 TransformerWrapper(exclude=None,
                                    include=['customer_id',
                                             'credit_lines_outstanding',
                                             'loan_amt_outstanding',
                                             'total_debt_outstanding', 'income',
                                             'years_employed', 'fico_score'],
                                    transformer=SimpleImputer(add_indicator=False,
                                                              copy=True,
                                                              fill_value=None,
                                                              keep_empty_features=False,
                                                              missing_values=n...
                                    transformer=StandardScaler(copy=True,
                                

In [22]:
import mlflow
# End run and get status
mlflow.end_run()